# Basile engine 2 · The published Babelia engine on its published constants
Basile's repository `librarianofbabel/libraryofbabel.info-algo` (CC BY-SA-NC, uploaded 2019-01-02) holds the image library's forward transform (`PRNG`, i.e. `babelia.cpp`) and its inverse (`imagesearch.cpp`). **The files are not part of ValaQuenta** (GPL-3.0-only); the engine reads them from a local cache at run time. To fetch them once (network, about 7 MB): `M.fetch_published()`.

Both programs are C++ with 3.19-million-bit integers; the engine runs them in Python integers (gmpy2 when present, plain Python otherwise: identical results).

In [1]:
import sys, os, importlib.util, math, time, random
sys.path.insert(0, os.path.abspath('../..'))
def load_maths():
    spec = importlib.util.spec_from_file_location("basile_maths", "../../modules/basile/maths.py")
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
M = load_maths()
CACHE = M.default_cache_dir()
HAVE = os.path.exists(os.path.join(CACHE, "PRNG")) and os.path.exists(os.path.join(CACHE, "imagesearch.cpp"))
print("engine loaded; published files in cache:", HAVE, "| gmpy2 accelerator:", M._gmpy2 is not None)
assert HAVE, 'published files not in the cache: run M.fetch_published() (network) and re-run'
pub = M.Published(CACHE)

engine loaded; published files in cache: True | gmpy2 accelerator: True


## The constants, audited
Everything below is measured on the parsed constants, not asserted.

In [2]:
au = pub.audit()
for k in ("bit_lengths", "shared_constants_identical", "a_times_ainverse_mod_m_is_1", "gcd_a_m_is_1", "gcd_c_m_is_1", "m_mod_4", "a_mod_4", "m_prime_power", "hull_dobell", "a_over_m_bits",
          "maskone_is_power_of_two", "masktwo_is_power_of_two", "divver_is_power_of_two", "chunk_bits", "drawn_bits_per_chunk", "image_bits", "m_bits", "palette_entries", "palette_is_hex_of_index"):
    print("%-32s %s" % (k, au[k]))
print("forward chain read from the PRNG file:", au["forward_steps"])
assert au["a_times_ainverse_mod_m_is_1"] and all(au["shared_constants_identical"].values())

bit_lengths                      {'m': 3194897, 'a': 1597519, 'c': 1628205, 'maskone': 2496010, 'masktwo': 1697290, 'divver': 399362, 'ainverse': 3194897}
shared_constants_identical       {'m': True, 'c': True, 'maskone': True, 'masktwo': True}
a_times_ainverse_mod_m_is_1      True
gcd_a_m_is_1                     True
gcd_c_m_is_1                     True
m_mod_4                          3
a_mod_4                          3
m_prime_power                    (3, 2015755)
hull_dobell                      {'gcd_m_c_is_1': True, 'a_minus_1_divisible_by_prime_factors': True, 'a_minus_1_divisible_by_4_if_4_divides_m': True, 'full_period': True}
a_over_m_bits                    -1597378
maskone_is_power_of_two          True
masktwo_is_power_of_two          True
divver_is_power_of_two           True
chunk_bits                       399361
drawn_bits_per_chunk             399360
image_bits                       3194880
m_bits                           3194897
palette_entries                  40

**Reading it.** `PRNG` and `imagesearch.cpp` carry the same `m, c, maskone, masktwo`; `a·a⁻¹ ≡ 1 (mod m)` holds; the three masks are exact powers of two, so `% maskone` is a bit mask and `% divver` keeps the low 399,361 bits. `m` is exactly a power of 3 (`3^2,015,755`), so Hull–Dobell is decidable: `gcd(c, m) = 1` and `3 | a − 1` hold, the published LCG alone is one cycle of length `m`. `a` and `c` are about half the size of `m` (the README advises values close to `m`). The palette table is exactly `#%03X` of its index.

## Geometry: what the image reads
640×416 = 8 quarters of 160×208 = 33,280 pixels each; 12 bits per pixel; each quarter is a 399,361-bit chunk of which 399,360 bits are drawn.

In [3]:
print("image bits          :", au["image_bits"], "= 640*416*12")
print("bits of m           :", au["m_bits"], "(", au["m_bits"] - au["image_bits"], "more than the image )")
print("bits the chunks span:", pub.chunk_bits * 8)
print("unread positions    :", pub.unread_positions())

image bits          : 3194880 = 640*416*12
bits of m           : 3194897 ( 17 more than the image )
bits the chunks span: 3194888
unread positions    : [399360, 798721, 1198082, 1597443, 1996804, 2396165, 2795526, 3194887, 3194888, 3194889, 3194890, 3194891, 3194892, 3194893, 3194894, 3194895, 3194896]


## Forward and inverse, on a random location

In [4]:
rnd = random.Random(1)
loc = rnd.getrandbits(au["m_bits"] - 2) % int(pub.m)
t0 = time.perf_counter(); p = pub.forward(loc); t1 = time.perf_counter(); back = pub.inverse(p); t2 = time.perf_counter()
print("forward %.2fs | inverse %.2fs | inverse(forward(loc)) == loc: %s" % (t1 - t0, t2 - t1, back == loc))
assert back == loc

forward 0.04s | inverse 0.05s | inverse(forward(loc)) == loc: True


## An image: pixels, and the location the search returns
The first five pixels of the first quarter at that location, as the site's `#RGB` codes.

In [5]:
quarters = pub.pixels(p)
print("quarters:", len(quarters), "| pixels each:", len(quarters[0]), "| first five:", ["#%03X" % c for c in quarters[0][:5]])
loc2 = pub.search(quarters)
same = pub.pixels(pub.forward(loc2)) == quarters
print("search(image) returns a location whose image is this image:", same, "| that location equals the one we started from:", loc2 == loc)

quarters: 8 | pixels each: 33280 | first five: ['#0B5', '#5A2', '#EE1', '#01B', '#A3D']
search(image) returns a location whose image is this image: True | that location equals the one we started from: False


**The search returns a location, not the location.** The image reads fewer bits than the location space holds (17 unread positions), so several locations show one image; the search sets the unread bits to zero and returns one of them.

## Several locations, one image
For each unread bit position: flip it in the transformed value, invert, and check whether a *different* location gives the *same* image.

In [6]:
print("position   distinct-location  forward(location)==value  same-image")
for pos in pub.unread_positions():
    w = pub.sibling(loc, pos)
    print("%-10d %-18s %-25s %s" % (pos, w["sibling_differs"], w["forward_matches"], w["same_image"]))
print(pub.mean_locations_per_image())

position   distinct-location  forward(location)==value  same-image
399360     True               True                      True
798721     True               True                      True
1198082    True               True                      True
1597443    True               True                      True
1996804    True               True                      True
2396165    True               True                      True
2795526    True               True                      True
3194887    True               True                      True
3194888    True               True                      True
3194889    True               True                      True
3194890    True               True                      True
3194891    True               True                      True
3194892    True               True                      True
3194893    True               True                      True
3194894    True               True                      True
3194895    True   

Sixteen of the seventeen unread positions give a second, valid location for the same image. The seventeenth, the highest, produces a value that no location transforms to (`forward_matches` is False), so it is not a witness. The average number of locations per image is `m / 2^3,194,880`, about 6.95×10⁴ (2¹⁶ × 1.06).

## The fibre of an image: additive within a location, different between locations
For each valid unread bit the offset from a location to its sibling is `inverse(forward(L) ^ 2^b) - L (mod m)`.

In [7]:
rnd = random.Random(9)
la = rnd.getrandbits(au["m_bits"] - 2) % int(pub.m); lb = rnd.getrandbits(au["m_bits"] - 2) % int(pub.m)
a = pub.sibling_offsets(la, pairs=6, seed=1)
print({k: v for k, v in a.items() if k != "offset_bit_lengths"})
pos = pub.unread_positions()[0]
o = [(pub.inverse(pub.forward(L) ^ (1 << pos)) - L) % int(pub.m) for L in (la, lb)]
print("same bit, two locations: offsets equal:", o[0] == o[1])
assert a["pairs_additive"] and a["all_bits_together_additive"] and o[0] != o[1]

{'valid_unread_bits': 17, 'single_offsets_distinct': True, 'pairs_tested': 6, 'pairs_additive': True, 'all_bits_together_additive': True, 'fibre_size_if_all_combinations_valid': 131072}
same bit, two locations: offsets equal: False


Within a location the sixteen offsets add (six random pairs and all sixteen together), so the fibre is `{L + a subset sum of the offsets}`, a box of 2¹⁶ points in Z_m. The offset for a given bit **differs** between the two locations: the generators of the box depend on the location, so the fibre is not a fixed coset of a subgroup. The Fourier transform of the fibre's indicator is `e^(2πikL/m) · ∏_b (1 + e^(2πik·o_b/m))`.

## What this notebook does not establish
* Whether the live site serves images by exactly this code: only the published source was read.
* The text library: its algorithm is not published.